In [ ]:
# Cell 1: Install dependencies
!pip install -q scanpy anndata igraph leidenalg scikit-learn scipy cellxgene-census umap-learn

In [ ]:
# Cell 2: Mount Drive and configure paths
from google.colab import drive
drive.mount('/content/drive')

import os, json, re
import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 10})

VOCAB_DIR      = '/content/drive/MyDrive/CellJEPA_results/universal_vocab/'
MT_DIR         = '/content/drive/MyDrive/CellJEPA_results/multitissue_universal/'
DATA_CACHE_DIR = '/content/drive/MyDrive/CellJEPA_results/data_cache/'
DIAG_DIR       = '/content/drive/MyDrive/CellJEPA_results/diagnostics/'

os.makedirs(DIAG_DIR, exist_ok=True)

VOCAB_FILE = os.path.join(VOCAB_DIR, 'universal_gene_names.json')

TISSUES = ['kidney', 'lung', 'liver', 'brain', 'heart', 'intestine']
TISSUE_COLORS = {
    'kidney':    '#4C72B0',
    'lung':      '#DD8452',
    'liver':     '#55A868',
    'brain':     '#C44E52',
    'heart':     '#8172B2',
    'intestine': '#937860',
}

print('Drive mounted.')
print(f'Vocab:       {VOCAB_FILE}')
print(f'Data cache:  {DATA_CACHE_DIR}')
print(f'Diagnostics: {DIAG_DIR}')

In [ ]:
# Cell 3: Load universal vocab + cached tissue arrays
# Requires Cell 6 of UniversalVocab_Colab.ipynb to have been run with --data_cache_dir set.
# Cached files are named: {tissue}_{n_cells}cells_universal.npy

with open(VOCAB_FILE) as f:
    universal_gene_names = json.load(f)
n_vocab = len(universal_gene_names)
print(f'Universal vocab: {n_vocab:,} genes')

tissue_arrays = {}

if not os.path.exists(DATA_CACHE_DIR):
    print(f'WARNING: DATA_CACHE_DIR not found: {DATA_CACHE_DIR}')
    print('Run Cell 6 of UniversalVocab_Colab.ipynb with --data_cache_dir set first.')
else:
    available = os.listdir(DATA_CACHE_DIR)
    for tissue in TISSUES:
        candidates = [
            f for f in available
            if f.startswith(tissue) and f.endswith('cells_universal.npy')
        ]
        if candidates:
            def _extract_n(fname):
                m = re.search(r'(\d+)cells', fname)
                return int(m.group(1)) if m else 0
            best = max(candidates, key=_extract_n)
            path = os.path.join(DATA_CACHE_DIR, best)
            tissue_arrays[tissue] = np.load(path)
            print(f'  {tissue:<10s}: {tissue_arrays[tissue].shape[0]:>7,} cells '
                  f'x {tissue_arrays[tissue].shape[1]:,} genes  (from {best})')
        else:
            print(f'  {tissue:<10s}: WARNING — no cache found')

total_cells = sum(X.shape[0] for X in tissue_arrays.values())
print(f'\nLoaded {len(tissue_arrays)}/{len(TISSUES)} tissues')
print(f'Total pre-training cells: {total_cells:,}')

In [ ]:
# Cell 4: Section A1 — Expressed genes per cell (per tissue + combined)
# Red dashed line = L_max=600. Cells to the left of this line will be fully used;
# cells to the right will have their genes randomly subsampled to 600 per forward pass.
# Orange dotted line = per-tissue median.

all_expressed_genes = {}
for tissue in TISSUES:
    if tissue in tissue_arrays:
        X = tissue_arrays[tissue]
        all_expressed_genes[tissue] = (X > 0).sum(axis=1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('A1: Expressed Genes Per Cell by Tissue', fontsize=13)

for i, tissue in enumerate(TISSUES):
    if tissue not in all_expressed_genes:
        axes.flat[i].set_visible(False)
        continue
    exp = all_expressed_genes[tissue]
    pct_ge600 = (exp >= 600).mean() * 100

    ax = axes.flat[i]
    ax.hist(exp, bins=60, color=TISSUE_COLORS[tissue], alpha=0.75,
            edgecolor='white', linewidth=0.2)
    ax.axvline(600, color='red', linestyle='--', lw=1.5, label='L_max=600')
    ax.axvline(np.median(exp), color='orange', linestyle=':', lw=1.4,
               label=f'median={np.median(exp):.0f}')
    ax.set_title(f'{tissue.capitalize()} ({pct_ge600:.0f}% ≥600 genes)', fontsize=9)
    ax.set_xlabel('Expressed genes / cell', fontsize=8)
    ax.set_ylabel('Cells', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if i == 0:
        ax.legend(fontsize=7)

# Combined distribution
if all_expressed_genes:
    combined = np.concatenate(list(all_expressed_genes.values()))
    pct_below = (combined < 600).mean() * 100
    ax = axes.flat[6]
    ax.hist(combined, bins=80, color='#2d6a4f', alpha=0.8, edgecolor='white', linewidth=0.2)
    ax.axvline(600, color='red', linestyle='--', lw=1.5, label='L_max=600')
    ax.axvline(np.median(combined), color='orange', linestyle=':', lw=1.4,
               label=f'median={np.median(combined):.0f}')
    ax.text(0.97, 0.95, f'{pct_below:.1f}% cells have\n<600 expressed genes',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_title(f'All tissues combined ({total_cells:,} cells)', fontsize=9)
    ax.set_xlabel('Expressed genes / cell', fontsize=8)
    ax.set_ylabel('Cells', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=7)

axes.flat[7].set_visible(False)
plt.tight_layout()
out = os.path.join(DIAG_DIR, 'A1_expressed_genes_per_cell.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

# Summary table
print('\n--- A1 Summary: Expressed genes per cell (universal vocab space) ---')
print(f'{"Tissue":<12} {"N cells":>8} {"Median":>8} {"Mean":>8} {"Min":>6} {"Max":>6} {"% ≥600":>7}')
print('-' * 60)
for tissue, exp in all_expressed_genes.items():
    print(f'{tissue:<12} {len(exp):>8,} {np.median(exp):>8.0f} {exp.mean():>8.1f} '
          f'{exp.min():>6} {exp.max():>6} {(exp>=600).mean()*100:>7.1f}')
if all_expressed_genes:
    comb = np.concatenate(list(all_expressed_genes.values()))
    print(f'{"TOTAL":<12} {len(comb):>8,} {np.median(comb):>8.0f} {comb.mean():>8.1f} '
          f'{comb.min():>6} {comb.max():>6} {(comb>=600).mean()*100:>7.1f}')

In [ ]:
# Cell 5: Section A2 — Sparsity and gene coverage diagnostics
# Left: per-cell sparsity distribution (fraction of zero genes) per tissue.
# Middle: expression distribution across all vocab genes (identifies dead/low-info genes).
# Right: fraction of universal vocab genes expressed in each tissue.

tissues_with_data = [t for t in TISSUES if t in tissue_arrays]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('A2: Sparsity and Gene Coverage Diagnostics', fontsize=12)

# Plot 1: Per-cell sparsity (fraction zeros) distribution
ax = axes[0]
for tissue in tissues_with_data:
    X = tissue_arrays[tissue]
    sparsity = (X == 0).mean(axis=1)
    ax.hist(sparsity, bins=50, alpha=0.55, color=TISSUE_COLORS[tissue],
            label=tissue, edgecolor='white', linewidth=0.2, density=True)
ax.set_xlabel('Fraction of zero genes per cell')
ax.set_ylabel('Density')
ax.set_title('Per-cell sparsity (fraction zeros)')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Plot 2: Mean expression per vocab gene (across all tissues combined)
ax = axes[1]
if tissue_arrays:
    # Process in tissue chunks to avoid OOM
    sum_expr = np.zeros(n_vocab, dtype=np.float64)
    any_expr  = np.zeros(n_vocab, dtype=bool)
    total_n = 0
    for tissue in tissues_with_data:
        X = tissue_arrays[tissue]
        sum_expr  += X.sum(axis=0)
        any_expr  |= (X > 0).any(axis=0)
        total_n   += X.shape[0]
    mean_expr = sum_expr / total_n
    n_expressed_genes = any_expr.sum()
    expressed_means = mean_expr[mean_expr > 0]
    ax.hist(expressed_means, bins=100, color='#4C72B0', alpha=0.8,
            edgecolor='white', linewidth=0.2)
    ax.set_xlabel('Mean log1p expression (across all cells)')
    ax.set_ylabel('Number of genes')
    ax.set_title(f'Expressed vocab genes: {n_expressed_genes:,}/{n_vocab:,}\n'
                 f'({n_expressed_genes/n_vocab*100:.1f}% of universal vocab expressed)')
    ax.set_yscale('log')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Plot 3: % vocab genes expressed per tissue
ax = axes[2]
pct_covered = [
    (tissue_arrays[t] > 0).any(axis=0).mean() * 100
    for t in tissues_with_data
]
colors_plot = [TISSUE_COLORS[t] for t in tissues_with_data]
bars = ax.bar(tissues_with_data, pct_covered, color=colors_plot, alpha=0.85)
for bar, pct in zip(bars, pct_covered):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('% universal vocab genes expressed\n(in ≥1 cell)')
ax.set_title('Gene coverage per tissue')
ax.set_ylim(0, max(pct_covered) * 1.18 + 1)
ax.tick_params(axis='x', rotation=30)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
out = os.path.join(DIAG_DIR, 'A2_sparsity_and_coverage.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

print('\n--- A2 Summary: Gene coverage per tissue ---')
print(f'{"Tissue":<12} {"Covered genes":>14} {"Coverage %":>12}')
print('-' * 40)
for tissue, pct in zip(tissues_with_data, pct_covered):
    n_cov = int(pct / 100 * n_vocab)
    print(f'{tissue:<12} {n_cov:>14,} {pct:>12.1f}')

In [ ]:
# Cell 6: Section B — HVG overlap between pre-training tissues
# Uses top-2000 HVGs (Seurat dispersion method) computed on each tissue's
# log1p-normalized cached array. Jaccard similarity measures how much the
# informative gene sets agree across tissues — higher overlap = more shared signal.
#
# NOTE: This cell requires ~3-4 GB RAM per tissue. On Colab A100 (83 GB) all
# 6 tissues can run simultaneously. On T4 (16 GB), process one at a time.

N_HVG = 2000

print(f'Computing top-{N_HVG} HVGs per tissue (Seurat dispersion method)...')
tissue_hvgs = {}  # tissue -> set of universal vocab indices

for tissue in TISSUES:
    if tissue not in tissue_arrays:
        continue
    X = tissue_arrays[tissue]
    adata_t = ad.AnnData(X=X.copy())
    adata_t.var_names = [str(g) for g in universal_gene_names]
    try:
        sc.pp.highly_variable_genes(adata_t, n_top_genes=N_HVG, flavor='seurat')
        hvg_mask = adata_t.var['highly_variable'].values
        tissue_hvgs[tissue] = set(np.where(hvg_mask)[0])
        print(f'  {tissue}: {len(tissue_hvgs[tissue])} HVGs')
    except Exception as e:
        print(f'  {tissue}: WARNING — HVG computation failed ({e})')
    del adata_t

# Pairwise Jaccard similarity and overlap counts
tissues_ok = [t for t in TISSUES if t in tissue_hvgs]
n = len(tissues_ok)
jaccard_matrix = np.zeros((n, n))
overlap_counts = np.zeros((n, n))

for i, t1 in enumerate(tissues_ok):
    for j, t2 in enumerate(tissues_ok):
        s1, s2 = tissue_hvgs[t1], tissue_hvgs[t2]
        inter = len(s1 & s2)
        union = len(s1 | s2)
        jaccard_matrix[i, j] = inter / union if union > 0 else 0.0
        overlap_counts[i, j] = inter

# --- Plots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'B: HVG Overlap Between Pre-training Tissues (top-{N_HVG} HVGs)', fontsize=12)

tick_labels = [t.capitalize() for t in tissues_ok]

# Jaccard heatmap
ax = axes[0]
im = ax.imshow(jaccard_matrix, cmap='Blues', vmin=0, vmax=jaccard_matrix.max())
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(tick_labels, rotation=45, ha='right')
ax.set_yticklabels(tick_labels)
for i in range(n):
    for j in range(n):
        color = 'white' if jaccard_matrix[i, j] > 0.6 * jaccard_matrix.max() else 'black'
        ax.text(j, i, f'{jaccard_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8, label='Jaccard similarity')
ax.set_title('B1: Jaccard Similarity')

# Overlap count heatmap
ax = axes[1]
im2 = ax.imshow(overlap_counts, cmap='YlOrRd')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(tick_labels, rotation=45, ha='right')
ax.set_yticklabels(tick_labels)
for i in range(n):
    for j in range(n):
        color = 'white' if overlap_counts[i, j] > 0.7 * overlap_counts.max() else 'black'
        ax.text(j, i, f'{int(overlap_counts[i,j])}', ha='center', va='center',
                fontsize=9, color=color)
plt.colorbar(im2, ax=ax, shrink=0.8, label='Number of shared HVGs')
ax.set_title('B2: Shared HVG Count')

plt.tight_layout()
out = os.path.join(DIAG_DIR, 'B_hvg_overlap_between_tissues.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

# Summary
print('\n--- B Summary: Pairwise HVG Jaccard similarity ---')
for i, t1 in enumerate(tissues_ok):
    for j, t2 in enumerate(tissues_ok):
        if i < j:
            print(f'  {t1} ∩ {t2}: {int(overlap_counts[i,j])} genes, Jaccard={jaccard_matrix[i,j]:.3f}')

In [ ]:
# Cell 7: Section C — PBMC-10k fine-tuning data quality
# Loads PBMC-10k via CELLxGENE Census, projects to universal vocab, applies HVG selection,
# then visualizes with PCA-based UMAP and shows the cell type distribution.
#
# Upload run_transfer_universal.py to /content/ before running this cell.

import sys
sys.path.insert(0, '/content')

from run_transfer_universal import load_pbmc10k_universal

print('Loading PBMC-10k (universal vocab, 2000 HVGs)...')
X_pbmc, int_labels, label_names, gene_to_col, _ = load_pbmc10k_universal(
    universal_gene_names, n_cells_subset=None, n_hvg=2000
)
print(f'PBMC-10k: {X_pbmc.shape[0]:,} cells x {X_pbmc.shape[1]:,} genes (universal vocab)')
print(f'Cell types ({len(label_names)}): {label_names}')

expressed_pbmc = (X_pbmc > 0).sum(axis=1)
print(f'\nExpressed genes per cell: median={np.median(expressed_pbmc):.0f}, '
      f'mean={expressed_pbmc.mean():.1f}, min={expressed_pbmc.min()}, max={expressed_pbmc.max()}')
print(f'{(expressed_pbmc >= 600).mean()*100:.1f}% of PBMC-10k cells have ≥600 expressed genes')

In [ ]:
# Cell 8: Section C (cont.) — PBMC-10k UMAP + cell type distribution
# Uses PCA (50 PCs) for dimensionality reduction before UMAP.
# This serves as a reference: the UMAP should show clean, well-separated clusters.
# If it doesn't, the issue is in the raw data, not the model.

print('Building AnnData and computing PCA + UMAP...')
adata_pbmc = ad.AnnData(X=X_pbmc)
adata_pbmc.obs['cell_type'] = [label_names[i] for i in int_labels]

sc.pp.pca(adata_pbmc, n_comps=50)
sc.pp.neighbors(adata_pbmc, n_neighbors=15, use_rep='X_pca')
sc.tl.umap(adata_pbmc)
print('Done.')

# Color palette
n_types = len(label_names)
palette = [
    '#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860',
    '#DA8BC3', '#8C8C8C', '#CCB974', '#64B5CD', '#E377C2', '#7F7F7F',
][:n_types]
type_to_color = {ct: palette[i] for i, ct in enumerate(label_names)}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# UMAP
ax = axes[0]
for ct in label_names:
    mask = adata_pbmc.obs['cell_type'] == ct
    ax.scatter(
        adata_pbmc.obsm['X_umap'][mask, 0],
        adata_pbmc.obsm['X_umap'][mask, 1],
        s=3, alpha=0.5, color=type_to_color[ct], label=ct,
    )
ax.set_title('C1: PBMC-10k UMAP (PCA 50 PCs)\nColored by ground-truth cell type', fontsize=10)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(markerscale=4, fontsize=7, bbox_to_anchor=(1.02, 1),
          loc='upper left', borderaxespad=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Cell type bar chart (sorted by count)
ax = axes[1]
counts = [(ct, (int_labels == i).sum()) for i, ct in enumerate(label_names)]
counts.sort(key=lambda x: -x[1])
ct_names  = [c[0] for c in counts]
ct_counts = [c[1] for c in counts]
ct_colors = [type_to_color[ct] for ct in ct_names]

bars = ax.barh(ct_names, ct_counts, color=ct_colors, alpha=0.85)
for bar, count in zip(bars, ct_counts):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
            f'{count:,}', va='center', fontsize=8)
ax.set_xlabel('Number of cells')
ax.set_title(f'C2: PBMC-10k Cell Type Distribution\n({X_pbmc.shape[0]:,} total cells)', fontsize=10)
ax.invert_yaxis()
ax.set_xlim(0, max(ct_counts) * 1.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
out = os.path.join(DIAG_DIR, 'C_pbmc10k_quality.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Cell 9: Section D — Transfer signal: pre-training HVG overlap with PBMC-10k HVGs
# For each pre-training tissue, computes how many of its top-2000 HVGs overlap
# with the top-2000 HVGs of PBMC-10k. Higher Jaccard = better transfer signal.
# Also shows the PBMC-10k HVGs (in universal vocab space) for reference.

# Compute PBMC-10k HVGs in universal vocab space
print('Computing PBMC-10k HVGs in universal vocab space...')
adata_pbmc_hvg = ad.AnnData(X=X_pbmc.copy())
adata_pbmc_hvg.var_names = [str(g) for g in universal_gene_names]
sc.pp.highly_variable_genes(adata_pbmc_hvg, n_top_genes=N_HVG, flavor='seurat')
pbmc_hvg_indices = set(np.where(adata_pbmc_hvg.var['highly_variable'].values)[0])
print(f'PBMC-10k: {len(pbmc_hvg_indices)} HVGs (in universal vocab)')
del adata_pbmc_hvg

# Compute overlap with each pre-training tissue
overlap_rows = []
for tissue in TISSUES:
    if tissue not in tissue_hvgs:
        continue
    s_tissue = tissue_hvgs[tissue]
    inter = len(s_tissue & pbmc_hvg_indices)
    union = len(s_tissue | pbmc_hvg_indices)
    jaccard = inter / union if union > 0 else 0.0
    overlap_rows.append((tissue, inter, jaccard))

print('\n--- D Summary: HVG overlap with PBMC-10k (transfer signal) ---')
print(f'{"Tissue":<12} {"Overlap":>10} {"/ PBMC-10k HVGs":>17} {"Jaccard":>10}')
print('-' * 52)
for tissue, inter, jac in overlap_rows:
    print(f'{tissue:<12} {inter:>10,} {inter/N_HVG*100:>16.1f}% {jac:>10.3f}')

# --- Plots ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'D: Transfer Signal — Pre-training HVG Overlap with PBMC-10k (top-{N_HVG} HVGs)',
             fontsize=11)

tissues_d  = [r[0] for r in overlap_rows]
jaccards_d = [r[2] for r in overlap_rows]
overlaps_d = [r[1] for r in overlap_rows]
colors_d   = [TISSUE_COLORS[t] for t in tissues_d]

ax = axes[0]
bars = ax.bar(tissues_d, jaccards_d, color=colors_d, alpha=0.85)
for bar, jac in zip(bars, jaccards_d):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{jac:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Jaccard(tissue HVGs, PBMC-10k HVGs)')
ax.set_title(f'D1: Jaccard similarity with PBMC-10k')
ax.set_ylim(0, max(jaccards_d) * 1.25 + 0.01)
ax.tick_params(axis='x', rotation=30)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[1]
bars = ax.bar(tissues_d, overlaps_d, color=colors_d, alpha=0.85)
for bar, ov in zip(bars, overlaps_d):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            f'{ov}', ha='center', va='bottom', fontsize=9)
ax.axhline(N_HVG, color='red', linestyle='--', lw=1.5, alpha=0.7, label=f'Max ({N_HVG})')
ax.set_ylabel(f'Number of shared HVGs (out of {N_HVG})')
ax.set_title('D2: Shared HVG count with PBMC-10k')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=30)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
out = os.path.join(DIAG_DIR, 'D_transfer_signal_hvg_overlap.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Cell 10: Section E — Training curves from pre-training logs
# Parses saved log files for l_jepa, l_rec, and cos_sim(student, teacher).
# cos_sim approaching 1.0 early in training is the collapse diagnostic.
#
# TO SAVE LOGS FROM CELL 6 of UniversalVocab_Colab.ipynb, replace the
# streaming block with:
#
#   LOG_PATH = os.path.join(MT_DIR, 'pretrain_jepa_log.txt')
#   with open(LOG_PATH, 'w') as logf:
#       for line in iter(proc.stdout.readline, ''):
#           print(line, end='', flush=True)
#           logf.write(line)
#
# Expected step log line format:
#   "l_jepa=X.XXXX | l_rec=X.XXXX | cos_sim(student,teacher)=X.XXXX"
# Expected epoch summary line format:
#   "Epoch N  l_jepa=X.XXXX  l_rec=X.XXXX  cos_sim=X.XXXX"

import re

STEP_PAT  = re.compile(
    r'l_jepa=([0-9.]+)\s*\|\s*l_rec=([0-9.]+)\s*\|\s*cos_sim\(student,teacher\)=([0-9.]+)'
)
EPOCH_PAT = re.compile(
    r'Epoch\s+(\d+).*?l_jepa=([0-9.]+)\s+l_rec=([0-9.]+)\s+cos_sim=([0-9.]+)'
)

def parse_pretrain_log(path):
    step_records, epoch_records = [], []
    with open(path) as f:
        for line in f:
            m = STEP_PAT.search(line)
            if m:
                step_records.append({
                    'l_jepa': float(m.group(1)),
                    'l_rec':  float(m.group(2)),
                    'cos_sim': float(m.group(3)),
                })
            m2 = EPOCH_PAT.search(line)
            if m2:
                epoch_records.append({
                    'epoch':   int(m2.group(1)),
                    'l_jepa':  float(m2.group(2)),
                    'l_rec':   float(m2.group(3)),
                    'cos_sim': float(m2.group(4)),
                })
    return step_records, epoch_records


LOG_PATHS = {
    'Cell-JEPA': os.path.join(MT_DIR, 'pretrain_jepa_log.txt'),
    'SIGReg':    os.path.join(MT_DIR, 'pretrain_sigreg_log.txt'),
}

found_logs = {name: p for name, p in LOG_PATHS.items() if os.path.exists(p)}

if not found_logs:
    print('No pre-training log files found.')
    print(f'Expected locations:')
    for name, p in LOG_PATHS.items():
        print(f'  {name}: {p}')
    print()
    print('To save logs from Cell 6, modify the streaming code as described in the cell comment above.')
else:
    for model_name, log_path in found_logs.items():
        step_records, epoch_records = parse_pretrain_log(log_path)
        print(f'{model_name}: {len(step_records)} step records, {len(epoch_records)} epoch records')

        if not step_records:
            print(f'  WARNING: no step records parsed from {log_path}')
            continue

        steps     = list(range(len(step_records)))
        l_jepas   = [r['l_jepa']  for r in step_records]
        l_recs    = [r['l_rec']   for r in step_records]
        cos_sims  = [r['cos_sim'] for r in step_records]

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(f'E: {model_name} Pre-training Curves', fontsize=12)

        ax = axes[0]
        ax.plot(steps, l_jepas, color='#C44E52', alpha=0.8, lw=1.0)
        if epoch_records:
            epoch_steps = [int(r['epoch'] / max(r2['epoch'] for r2 in epoch_records) *
                              len(steps)) for r in epoch_records]
            ax.scatter(epoch_steps, [r['l_jepa'] for r in epoch_records],
                       color='darkred', zorder=5, s=40, label='epoch end')
            ax.legend(fontsize=8)
        ax.set_xlabel('Step')
        ax.set_ylabel('JEPA loss (l_jepa)')
        ax.set_title('E1: JEPA Loss')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        ax = axes[1]
        ax.plot(steps, l_recs, color='#4C72B0', alpha=0.8, lw=1.0)
        ax.set_xlabel('Step')
        ax.set_ylabel('Reconstruction loss (l_rec)')
        ax.set_title('E2: Reconstruction Loss')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        ax = axes[2]
        ax.plot(steps, cos_sims, color='#55A868', alpha=0.8, lw=1.0)
        ax.axhline(1.0, color='red', linestyle='--', lw=1.3, alpha=0.7,
                   label='Collapse (=1.0)')
        ax.axhline(0.0, color='grey', linestyle=':', lw=1.0, alpha=0.5,
                   label='Random (=0.0)')
        ax.set_ylim(-0.1, 1.1)
        ax.set_xlabel('Step')
        ax.set_ylabel('cos_sim(student, teacher)')
        ax.set_title('E3: Collapse Diagnostic\n(→1.0 = collapse; healthy = gradual rise)')
        ax.legend(fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        safe = model_name.lower().replace(' ', '_').replace('-', '')
        out = os.path.join(DIAG_DIR, f'E_training_curves_{safe}.png')
        plt.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {out}')

In [ ]:
# Cell 11: Save summary report and list all outputs
from datetime import datetime

timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_path = os.path.join(DIAG_DIR, f'diagnostics_summary_{timestamp}.txt')

lines = []
lines.append(f'CellJEPA Diagnostic Summary — {datetime.now().strftime("%Y-%m-%d %H:%M")}')
lines.append('=' * 64)

lines.append(f'\nUniversal vocab: {n_vocab:,} genes')

lines.append('\n--- A: Cell Distribution ---')
lines.append(f'{"Tissue":<12} {"N cells":>8} {"Median":>8} {"% ≥600":>7}')
for tissue, exp in all_expressed_genes.items():
    lines.append(f'{tissue:<12} {len(exp):>8,} {np.median(exp):>8.0f} '
                 f'{(exp>=600).mean()*100:>7.1f}')

lines.append('\n--- A2: Gene coverage per tissue ---')
for tissue in tissues_with_data:
    n_cov = int((tissue_arrays[tissue] > 0).any(axis=0).sum())
    lines.append(f'  {tissue}: {n_cov:,}/{n_vocab:,} universal genes expressed '
                 f'({n_cov/n_vocab*100:.1f}%)')

if 'tissues_ok' in dir() and 'jaccard_matrix' in dir():
    lines.append('\n--- B: HVG Jaccard similarity between tissues ---')
    for i, t1 in enumerate(tissues_ok):
        for j, t2 in enumerate(tissues_ok):
            if i < j:
                lines.append(f'  {t1} ∩ {t2}: {int(overlap_counts[i,j])} genes, '
                              f'Jaccard={jaccard_matrix[i,j]:.3f}')

if 'overlap_rows' in dir():
    lines.append('\n--- D: HVG overlap with PBMC-10k (transfer signal) ---')
    for tissue, inter, jac in overlap_rows:
        lines.append(f'  {tissue}: {inter}/{N_HVG} HVGs overlap, Jaccard={jac:.3f}')

with open(summary_path, 'w') as f:
    f.write('\n'.join(lines) + '\n')

print(f'Summary saved: {summary_path}')
print(f'\nAll files in {DIAG_DIR}:')
for fname in sorted(os.listdir(DIAG_DIR)):
    fpath = os.path.join(DIAG_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname}  ({size_kb:.0f} KB)')